In [1]:
import xarray as xr

%config InlineBackend.figure_format='retina'
xr.set_options(display_style="text")
import os 
import pandas as pd

 Heatwaves are defined by 5 consecutives days of extreme heat.
 1) compute the extreme events for each day 
 2) if there is more than 5 --> heatwaves

In [2]:
ds = xr.open_zarr('../data/sst_daily.zarr',chunks={"time": 365, "lat": 180,"lon": 360})

/var/folders/k8/8bd2c6nd31s1yp4rsc4gj7340000gn/T/ipykernel_27670/574942760.py:1: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_zarr('../data/sst_daily.zarr',chunks={"time": 365, "lat": 180,"lon": 360})


In [3]:
ds = ds.coarsen(lat=2, lon=2).mean()


In [4]:
da = ds.sst

In [5]:
da.shape

(15097, 360, 720)

In [6]:
sst_gb = da.groupby("time.dayofyear")

In [7]:
clim_dayofyear = sst_gb.mean(dim='time')
# what is the temperature on average for each calendar year

In [8]:
clim_dayofyear

<xarray.DataArray 'sst' (dayofyear: 366, lat: 360, lon: 720)> Size: 379MB
dask.array<transpose, shape=(366, 360, 720), dtype=float32, chunksize=(366, 90, 180), chunktype=numpy.ndarray>
Coordinates:
  * dayofyear  (dayofyear) int64 3kB 1 2 3 4 5 6 7 ... 361 362 363 364 365 366
  * lat        (lat) float32 1kB -89.75 -89.25 -88.75 ... 88.75 89.25 89.75
  * lon        (lon) float32 3kB 0.25 0.75 1.25 1.75 ... 358.2 358.8 359.2 359.8
Attributes:
    long_name:     Daily Sea Surface Temperature
    units:         degC
    valid_range:   [-3.0, 45.0]
    precision:     2.0
    dataset:       NOAA High-resolution Blended Analysis
    var_desc:      Sea Surface Temperature
    level_desc:    Surface
    statistic:     Mean
    parent_stat:   Individual Observations
    actual_range:  [-1.7999999523162842, 36.56999969482422]

In [9]:
# clim_dayofyear[:,250,0].plot()

In [10]:
# Compute threshold using a +/-5 day window across all years
rolling_window = 11  # 5 days before + current + 5 days after
window_radius = rolling_window // 2

da = da.assign_coords(
    year=da['time'].dt.year,
    dayofyear=da['time'].dt.dayofyear,
)

In [11]:
# Reshape to (dayofyear, year, lat, lon)
sst_yd = (
    da
    .set_index(time=['year', 'dayofyear'])
    .drop_duplicates('time')
    .unstack('time')
    .transpose('dayofyear', 'year', ...)
)

# Circular pad on dayofyear to support window at year boundaries
pad_left = sst_yd.isel(dayofyear=slice(-window_radius, None))
pad_right = sst_yd.isel(dayofyear=slice(0, window_radius))
sst_pad = xr.concat([pad_left, sst_yd, pad_right], dim='dayofyear')

# Rolling window over dayofyear, then compute quantile across (year, window)
threshold = (
    sst_pad
    .rolling(dayofyear=rolling_window, center=True)
    .construct('window')
    .quantile(0.9, dim=('year', 'window'))
)

# Keep the real dayofyear range (1..366)
threshold = threshold.isel(dayofyear=slice(window_radius, window_radius + sst_yd.sizes['dayofyear']))
threshold


<xarray.DataArray 'sst' (dayofyear: 366, lat: 360, lon: 720)> Size: 759MB
dask.array<getitem, shape=(366, 360, 720), dtype=float64, chunksize=(138, 16, 33), chunktype=numpy.ndarray>
Coordinates:
  * dayofyear  (dayofyear) int64 3kB 1 2 3 4 5 6 7 ... 361 362 363 364 365 366
  * lat        (lat) float32 1kB -89.75 -89.25 -88.75 ... 88.75 89.25 89.75
  * lon        (lon) float32 3kB 0.25 0.75 1.25 1.75 ... 358.2 358.8 359.2 359.8
    quantile   float64 8B 0.9
Attributes:
    long_name:     Daily Sea Surface Temperature
    units:         degC
    valid_range:   [-3.0, 45.0]
    precision:     2.0
    dataset:       NOAA High-resolution Blended Analysis
    var_desc:      Sea Surface Temperature
    level_desc:    Surface
    statistic:     Mean
    parent_stat:   Individual Observations
    actual_range:  [-1.7999999523162842, 36.56999969482422]

In [12]:
ds.sst.data

dask.array<mean_agg-aggregate, shape=(15097, 360, 720), dtype=float32, chunksize=(365, 90, 180), chunktype=numpy.ndarray>

## Detect Marine Heatwave Events (>=5 consecutive days)
This marks all days that belong to a run of exceedances with length >= MIN_DURATION.


In [13]:
import numpy as np

MIN_DURATION = 5

# Align threshold (dayofyear, lat, lon) with full time series
thresh_time = threshold.sel(dayofyear=da['time'].dt.dayofyear)

# Boolean exceedance array over time
exceed = (da > thresh_time)

# Mark full runs of consecutive True values with length >= MIN_DURATION

def mark_runs_1d(arr, min_len):
    arr = np.asarray(arr, dtype=bool)
    n = arr.size
    out = np.zeros(n, dtype=bool)
    if n == 0:
        return out
    diff = np.diff(arr.astype(int))
    starts = np.where(diff == 1)[0] + 1
    ends = np.where(diff == -1)[0] + 1
    if arr[0]:
        starts = np.r_[0, starts]
    if arr[-1]:
        ends = np.r_[ends, n]
    for s, e in zip(starts, ends):
        if (e - s) >= min_len:
            out[s:e] = True
    return out

mhw_mask = xr.apply_ufunc(
    mark_runs_1d,
    exceed,
    kwargs={"min_len": MIN_DURATION},
    input_core_dims=[["time"]],
    output_core_dims=[["time"]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[bool],
    dask_gufunc_kwargs={'allow_rechunk':True}
)
mhw_mask

<xarray.DataArray 'sst' (lat: 360, lon: 720, time: 15097)> Size: 4GB
dask.array<transpose, shape=(360, 720, 15097), dtype=bool, chunksize=(2, 3, 15097), chunktype=numpy.ndarray>
Coordinates:
  * lat        (lat) float32 1kB -89.75 -89.25 -88.75 ... 88.75 89.25 89.75
  * lon        (lon) float32 3kB 0.25 0.75 1.25 1.75 ... 358.2 358.8 359.2 359.8
  * time       (time) datetime64[ns] 121kB 1981-09-01 1981-09-02 ... 2023-12-31
    year       (time) int64 121kB 1981 1981 1981 1981 ... 2023 2023 2023 2023
    dayofyear  (time) int64 121kB 244 245 246 247 248 ... 361 362 363 364 365
    quantile   float64 8B 0.9
Attributes:
    long_name:     Daily Sea Surface Temperature
    units:         degC
    valid_range:   [-3.0, 45.0]
    precision:     2.0
    dataset:       NOAA High-resolution Blended Analysis
    var_desc:      Sea Surface Temperature
    level_desc:    Surface
    statistic:     Mean
    parent_stat:   Individual Observations
    actual_range:  [-1.7999999523162842, 36.56999969482422]

In [14]:
land_mask = da.isnull().all("time")  # True where land

In [15]:
mhw_mask = mhw_mask.transpose('time', 'lat', 'lon')
mhw_days_per_year = mhw_mask.groupby("time.year").sum("time")
print(mhw_days_per_year.shape)

(41, 360, 720)


In [16]:
mhw_days_per_year = mhw_days_per_year.where(~land_mask)


In [17]:
mhw_days_per_year = mhw_days_per_year.chunk({'year':"auto", 'lon':'auto', 'lat':'auto'})

In [18]:
# Save a downsampled version to keep write time reasonable
mhw_days_per_year_ds = (
    mhw_days_per_year
    .coarsen(lat=2, lon=2, boundary="trim")
    .mean()
    .chunk({"year": -1, "lat": 90, "lon": 90})
)

mhw_days_per_year_ds.to_zarr(
    "../data/cache/mhw_days_per_year.zarr",
    mode="w",
    consolidated=True,
)


/Users/luciedole/miniconda3/envs/xarray-tutorial/lib/python3.13/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
def count_events_1d(arr, min_len=5):
    arr = np.asarray(arr, dtype=bool)
    if arr.size == 0:
        return 0
    diff = np.diff(arr.astype(int))
    starts = np.where(diff == 1)[0] + 1
    ends = np.where(diff == -1)[0] + 1
    if arr[0]:
        starts = np.r_[0, starts]
    if arr[-1]:
        ends = np.r_[ends, arr.size]
    return np.sum((ends - starts) >= min_len)

mhw_events_per_year = xr.apply_ufunc(
    count_events_1d,
    mhw_mask.groupby("time.year"),
    input_core_dims=[["time"]],
    output_core_dims=[[]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[int],
)

mhw_events_per_year

<xarray.DataArray 'sst' (lat: 180, lon: 360, year: 41)> Size: 21MB
dask.array<transpose, shape=(180, 360, 41), dtype=int64, chunksize=(2, 2, 1), chunktype=numpy.ndarray>
Coordinates:
  * lat       (lat) float32 720B -89.88 -88.88 -87.88 ... 87.12 88.12 89.12
  * lon       (lon) float32 1kB 0.125 1.125 2.125 3.125 ... 357.1 358.1 359.1
  * year      (year) int64 328B 1981 1982 1983 1984 1985 ... 2020 2021 2022 2023
    quantile  float64 8B 0.9
Attributes:
    long_name:     Daily Sea Surface Temperature
    units:         degC
    valid_range:   [-3.0, 45.0]
    precision:     2.0
    dataset:       NOAA High-resolution Blended Analysis
    var_desc:      Sea Surface Temperature
    level_desc:    Surface
    statistic:     Mean
    parent_stat:   Individual Observations
    actual_range:  [-1.7999999523162842, 36.56999969482422]

## Plot MHW Days per Year 


In [56]:
mhw_events_per_year = mhw_events_per_year.transpose('year', 'lat', 'lon')

In [67]:
mhw_events_per_year = mhw_events_per_year.where(~land_mask)

In [73]:
# Save a downsampled version to keep write time reasonable
mhw_events_per_year_ds = (
    mhw_events_per_year
    .coarsen(lat=4, lon=4, boundary="trim")
    .mean()
    .chunk({"year": -1, "lat": 90, "lon": 90})
)

mhw_events_per_year_ds.to_zarr(
    "../data/cache/mhw_events_per_year.zarr",
    mode="w",
    consolidated=True,
)
# save these two datasets


KeyboardInterrupt: 

In [71]:
mhw_plot = mhw_days_per_year.isel(year=-1)    

In [ ]:
mhw_plot.hvplot(
    x="lon",
    y="lat",
    width=800,
    height=400,
    cmap="inferno",
    clim=(0, float(mhw_plot.max()))
).opts(active_tools=["pan"])


In [69]:
mhw_plot = mhw_events_per_year.isel(year=-1)    

In [70]:
mhw_plot.hvplot(
    x="lon",
    y="lat",
    width=800,
    height=400,
    cmap="inferno",
    clim=(0, float(mhw_plot.max()))
).opts(active_tools=["pan"])


KeyboardInterrupt: 